# M3C4 — Level 1: Feature Selection Core

**Purpose:** compare all features with MI and RFE, then explain your recommendation.

**Do:** save a copy in Drive, run from top to bottom, and upload `telco_source.csv` and `M3C4_Runtime.zip` when asked. After a runtime reset, start again at C01.

**Check:** write your own C04 and C16 answers. Choose `top_k` before viewing validation results. The final test remains closed.

## C01 — 🚀 Start the workspace
**Purpose:** rebuild the five C3 features and protected split.

**Do:** run the cell below. If the picker opens, select **both** `telco_source.csv` and `M3C4_Runtime.zip`. The ZIP contains readable Python preparation code and the frozen split JSON. No installation or web download is needed.

**Check:** train **4225**, validation **1409**, final test reserved **1409**, and **50 columns**, followed by five real TRAIN rows without IDs. After a reset, start here again. Local Jupyter: keep both files beside your notebook.

In [ ]:
#@title ▶ Start the lesson
from pathlib import Path
from zipfile import ZipFile, BadZipFile
comparison = results = rfe_columns = None  # Invalidate old results before a new start.
required = ['telco_source.csv', 'M3C4_Runtime.zip']
if any(not Path(name).exists() for name in required):
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Place telco_source.csv and M3C4_Runtime.zip beside this notebook, then rerun C01.') from None
    print('Select BOTH telco_source.csv and M3C4_Runtime.zip from the lesson package.')
    files.upload()
missing = [name for name in required if not Path(name).exists()]
if missing:
    raise FileNotFoundError('Missing: ' + ', '.join(missing) + '. Rerun C01 with the issued files and original names.')
try:
    with ZipFile('M3C4_Runtime.zip') as support:
        folder = Path('_m3c4_runtime'); folder.mkdir(exist_ok=True)
        for name in ['m3c4_runtime.py', 'M3_Split_Card.json']:
            (folder / name).write_bytes(support.read(name))
except (BadZipFile, KeyError):
    raise ValueError('Wrong or damaged M3C4_Runtime.zip. Upload the issued ZIP again, then rerun C01.') from None
exec(compile((folder / 'm3c4_runtime.py').read_text(encoding='utf-8'), 'm3c4_runtime.py', 'exec'), globals())

## C02 — Predict before running
Toy rows: tenure = [0,12,24,36], tenure_years = [0,1,2,3]. Do the two columns add different information?

**My prediction:** …
The declared priority order is tenure first, tenure_years second. The filter keeps the earlier field when a pair exceeds the correlation threshold. Before running C03, say in your own words which field this rule would retain and why:

**My choice and reason:** …
Correlation r ranges from −1 to 1. abs(r)>0.85 includes strong reverse relationships. A zero correlation can still hide a nonlinear link.

## C03 — 🔎 Run the correlation check
**Do:** Compare the toy columns, then inspect the train heatmap.

**Check:** Find one pair with abs(r) > 0.85 and read its removal record.

In [ ]:
toy=pd.DataFrame({'tenure':[0,12,24,36],'tenure_years':[0,1,2,3],'fee':[30,50,20,40]})
show_table('Toy example · identical information in different units','Tenure and tenure_years move together exactly in this toy example.',toy.corr().rename_axis('Feature').reset_index())
assert toy.corr().loc['tenure','tenure_years']==1
correlation_report()

## C04 — My correlation decision
Read one actual pair and its removal record above.

**Pair and r:** …

**Removed field:** …

**Why keep the other one:** …

**What this filter cannot prove:** …

**How this prepared filter works:** it keeps the earlier simpler field in the declared source/formula order. A pair already covered by a prior removal needs no second deletion. `all` keeps the full pre-filter C3 set.

## C08 — 📊 Rank with MI
**Do:** Before running: four toy customers have Churn = [Yes, Yes, No, No]. A flag is [1, 1, 0, 0]; another column is [1, 1, 1, 1]. Which column can distinguish the outcomes? MI measures this kind of information, including nonlinear links. Now run the train-only ranking.

**Check:** Read the candidate count and discrete-column list before choosing top_k.

In [ ]:
# C08 — Prepared train-only MI ranking. No validation results yet.
mi_ranking()

## C09 — Choose BEFORE seeing validation
Read the ranking and pick an integer top_k from 1 to the printed candidate count. Here **cutoff = top_k, the NUMBER of highest-ranked features**, not an MI-score threshold. `num__tenure` means the numeric tenure field; `cat__Contract_Month-to-month` is the one-hot question “Is Contract Month-to-month?” More MI does not guarantee a better combination. Write your own reason and one selected feature you can explain. Do not tune repeatedly against validation; changing the choice invalidates downstream results. After any choice/code change rerun **C12 → C14 → C15** (run updated C11 first if its code changed). After reset start from **C01**; old JSON/output is stale.
References: https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.mutual_info_classif.html . One-hot columns AND the integer-count fields SeniorCitizen/num_services are discrete; other numeric fields are continuous.

In [ ]:
top_k = None
selection_reason = ''
# My selected feature and explanation (write in your own words): ...

## C11 — Provided learning code

**Purpose:** compare all 50 prepared features with your MI selection on the same rows.

**Do:** run this provided learning code; it was not authored by your AI agent.

**Check:** C12 independently checks the all and MI results.

In [ ]:
# Provided learning code — run this cell.
def compare_all_mi(X_train, X_validation, y_train, y_validation, mi_columns):
    rows=[]
    for label, columns in [('all',list(X_train.columns)),('MI',list(mi_columns))]:
        model=RandomForestClassifier(n_estimators=100,random_state=42)
        model.fit(X_train[columns],y_train)
        predictions=model.predict(X_validation[columns])
        rows.append({'Feature Set':label,'Num Features':len(columns),'Accuracy':accuracy_score(y_validation,predictions),'F1':f1_score(y_validation,predictions,pos_label='Yes')})
    return pd.DataFrame(rows)


## C12 — ✅ Validate all versus MI
**Do:** Read the scores, then run the check.

- **Accuracy = correct / all.** Toy: 8 correct out of 10 gives 0.80. This alone does not show how well we find leavers.
- **F1 balances finding leavers and avoiding false alarms.** It combines how many predicted leavers really leave with how many actual leavers we find. It ranges from 0 to 1; higher is better on this check.

Compare both scores. Keep the top_k you chose before validation.

**Check:** Two rows appear: all and MI. If it fails, read the error and ask the mentor for help.

In [ ]:
# Intermediate all versus MI; RFE joins in C15.
validate_compare()

## C14 — 🔁 Select with RFE
**Do:** If we wanted two inputs out of four, RFE would fit the model, remove one weak input, fit again, and stop at two. Here it keeps floor(half the candidate count), using RF100, seed42 and step=1. Run the prepared selector and wait for it to finish.

**Check:** Read Selected, Rank, MI ∩ RFE and measured runtime.

In [ ]:
# C14 — Prepared RFE: RF100 seed42, step=1, floor(half candidates), at least 1.
# https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html
run_rfe()

## C15 — 📋 Compare all three sets
**Do:** Run the final comparison.

**Check:** The table has all, MI and RFE on the same validation rows.

In [ ]:
# Same model and validation customers for all three sets.
final_table()

## C16 — My recommendation and transfer
**I recommend … because the actual Accuracy/F1 and counts show …**
**MI/RFE overlap and one disagreement, if any (otherwise say none):** …

**A small accuracy gap corresponds to how many of these 1409 customers?** …

**Why this is not causal proof:** …

**For a new dataset, what must be fitted again on train (or inside each C6 CV fold)?** …
**MI/RFE selected MonthlyCharges. A student says higher charges cause churn. What does your evidence support, and what cannot it prove?** …
**For a new train dataset, list what you must learn again for preprocessing, correlation decisions, MI and RFE; when would you choose top_k before viewing the new validation result?** …
Write your own analysis. There is no automatic claim that fewer features always win.

## Save and reopen

Save the notebook in Drive or download `.ipynb`. Reopen it and check your C04 and C16 answers and latest C15 table. After changing `top_k`, rerun C09, C11, C12, C14 and C15. After reset start from C01; old outputs can be stale.